# 04 KNN comparable bonds

Role 4. Due Friday 20 Nov (first k search on Thursday).

Price a bond from the spreads of its k most similar bonds, as a trader does with comparables: how well it works and where it breaks. Regression: `KNeighborsRegressor` and MAE.

Class notebooks to follow: `2. KNN_IBM attrition` (the structure) and `1. Manual & scikit KNN` (the distance).

This is a starter: the setup cell runs as it is, and each step of your walkthrough has a heading and an empty
cell below. Fill the cells in order, following the class notebook.

Rules: validation only (`partition == "valid"`); test is opened once, in `99_final`. Shared code comes from
`ml2` and is never copied here. Figures go to `reports/figures/04_<what>.png` with units on the axes
(bp, years). Every run, including failures and dropped ideas, gets one line in `RUN_LOG.md`.

## 1. Setup

KNN needs scaled features: `make_preprocessor()` scales inside the pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline

from ml2.baseline import matrix_same_date
from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.evaluate import metrics_by_stratum, regression_metrics
from ml2.features import make_preprocessor, xy
from ml2.splits import issuer_group_folds, walk_forward_folds

cfg = load_config()
TARGET = cfg["problem"]["target"]
FIG = resolve("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)

check_panel()
df = load_panel(partitions=["train", "valid"])
train, valid = df[df.partition == "train"], df[df.partition == "valid"]
X_tr, y_tr = xy(train)
X_va, y_va = xy(valid)

# Searches run on a 30% sample of train (time order kept); chosen models are refitted on all of train
sub = train.sample(frac=0.3, random_state=42).sort_values("snapshot_date")
X_s, y_s = xy(sub)
cv = walk_forward_folds(sub.loc[X_s.index, "snapshot_date"])  # replaces cv=5 / KFold in the class notebooks
print(f"train {X_tr.shape}, valid {X_va.shape}, search sample {X_s.shape}, {len(cv)} time-ordered folds")

In [ ]:
# The bar: matrix pricing re-marked on each date (the gate baseline), on valid. About 1.5 minutes.
valid = valid.copy()
valid["pred_matrix_same_date"] = matrix_same_date(valid)


def vs_bar(pred, name="model"):
    """MAE by grade for predictions on X_va's rows, next to the gate baseline on the same rows."""
    rows = valid.loc[X_va.index].assign(pred=np.asarray(pred))
    out = pd.concat({
        name: metrics_by_stratum(rows, TARGET, "pred", "grade_bucket").set_index("stratum")["mae_bps"],
        "matrix re-marked (bar)": metrics_by_stratum(
            rows, TARGET, "pred_matrix_same_date", "grade_bucket").set_index("stratum")["mae_bps"],
    }, axis=1).T
    return out[["IG", "HY", "ALL"]].round(1)


vs_bar(valid.loc[X_va.index, "pred_matrix_same_date"], "matrix re-marked")  # check: both rows equal

## 2. Standardization inside the pipeline

`make_knn(k, weights)`: Pipeline of `make_preprocessor()` and `KNeighborsRegressor`. Why scale, and why inside the pipeline.

In [ ]:
# Step 2: Standardization inside the pipeline

## 3. KNN model 1

k = 5, `weights="distance"`; `vs_bar(...)` on valid.

In [ ]:
# Step 3: KNN model 1

## 4. Look for the best k manually

CV MAE for k from 2 to 40, best k marked; the U shape.

In [ ]:
# Step 4: Look for the best k manually

## 5. KNN model 2: grid search

`GridSearchCV` over the number of neighbours and the weights, `cv=cv`.

In [ ]:
# Step 5: KNN model 2: grid search

## 6. KNN model 3: bagging of KNNs

`BaggingRegressor` of KNN after the preprocessor. Does it help, and why not much?

In [ ]:
# Step 6: KNN model 3: bagging of KNNs

## 7. What makes bonds comparable?

All features against a trader's short list: `years_to_maturity`, `coupon_pct`, `grade_bucket`, `sector`, `vol_252d`, `leverage`, `log_market_cap`.

In [ ]:
# Step 7: What makes bonds comparable?

## 8. Speed

Time the predictions on valid: fast to train, slow to predict.

In [ ]:
# Step 8: Speed

## 9. New-issuer question

The best setup with `issuer_group_folds(sub.loc[X_s.index, "issuer_ticker"])`.

In [ ]:
# Step 9: New-issuer question

## 10. Show the comparables

For 3 valid bonds, their 5 nearest training neighbours (`kneighbors`) with issuer, maturity, grade and spread.

In [ ]:
# Step 10: Show the comparables

## RUN_LOG

One line per run in `notebooks/RUN_LOG.md`:

`| date | who | notebook | model and key parameters | features | fit on train, scored on valid | MAE bps (IG / HY) | notes |`